# 07b — Nhật ký làm sạch (cleaning log) cho 8 thành phố

Theo yêu cầu của slide W4 (EX4.4) và W8: nhật ký phải dùng **mẫu `Step · Column · Finding · Action · Rows · Why`**, ghi **mọi quyết định**
(kể cả quyết định *giữ nguyên*), mỗi dòng nói rõ **lý do, cách kiểm chứng và rủi ro**.

Mọi con số trong bảng được **tính trực tiếp từ file gốc và file đã làm sạch** trong notebook này (không gõ tay), nên chạy lại là cập nhật theo.
Chạy sau notebook 07. Kết quả: `reports/cleaning_log_8cities.csv` (bản đầy đủ, thêm cột `City`) và `reports/cleaning_log_report_8cities.csv` (bản rút gọn 13 dòng để đưa vào báo cáo).

Quy ước: **Rows = số ô (ngày × cột) bị ảnh hưởng**; dòng áp cho nhiều thành phố có chi tiết từng thành phố trong `Finding`.
Phần `Why` có 3 ý khi cần: lý do · *Kiểm chứng* · *Rủi ro*.

In [1]:
import sys, os
sys.path.append(os.path.abspath("../src"))
import numpy as np, pandas as pd
from utils import flag_flatline_zeros

RAW, PROCESSED, REPORTS = "../data/raw/city_day.csv", "../data/processed", "../reports"
CITIES = ["Delhi", "Gurugram", "Lucknow", "Jaipur", "Bengaluru", "Amritsar", "Chennai", "Hyderabad"]
POL = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene"]
MAX_GAP, MISSING_MAX_PCT = 14, 60

raw = pd.read_csv(RAW, parse_dates=["Date"])
raw8 = raw[raw.City.isin(CITIES)]
m = pd.read_csv(f"{PROCESSED}/air_quality_8cities.csv", parse_dates=["Date"])
sus = pd.read_csv(f"{REPORTS}/suspect_points_8cities.csv", parse_dates=["date"])

def vn(x, nd=0):                       # số kiểu Việt Nam: 1.234,5
    s = f"{x:,.{nd}f}"
    return s.replace(",", "X").replace(".", ",").replace("X", ".")
def by_city(d, nd=0):                  # {'Delhi': 98, ...} -> 'Delhi 98, Gurugram 294, ...'
    return ", ".join(f"{c} {vn(v, nd)}" for c, v in d.items() if v)

LOG = []
def add(city, column, finding, action, rows, why):
    LOG.append(dict(Step=len(LOG) + 1, City=city, Column=column, Finding=finding, Action=action, Rows=int(rows), Why=why))

## A. Dữ liệu gốc và kiểm tra cấu trúc

In [2]:
n_by = raw8.groupby("City").size().reindex(CITIES)
start = raw8.groupby("City").Date.min().reindex(CITIES)
add("Tất cả", "(toàn bộ)",
    f"city_day.csv có {vn(len(raw))} dòng x {raw.shape[1]} cột, {raw.City.nunique()} thành phố; lấy 8 thành phố của nhóm = {vn(len(raw8))} dòng ({by_city(n_by)})",
    "Đọc file gốc, không sửa file raw", len(raw8),
    "File raw giữ nguyên; mọi thay đổi đi qua notebook 01 -> 07 nên chạy lại được từ đầu. "
    "Kiểm chứng: chạy lại 01 -> 07 từ file gốc trên một bản sao sạch cho ra đúng cùng bảng gộp (so từng ô, giống hệt).")

assert m.duplicated(["City", "Date"]).sum() == 0
cont = all((g.Date.max() - g.Date.min()).days + 1 == len(g) for _, g in m.groupby("City"))
add("Tất cả", "Date, City",
    f"0 dòng trùng (City, Date); chuỗi ngày liên tục, không thiếu ngày giữa chuỗi: {cont}. Độ dài khác nhau: "
    + ", ".join(f"{c} từ {start[c]:%d/%m/%Y}" for c in CITIES if start[c] > pd.Timestamp('2015-01-01')) + "; còn lại từ 01/01/2015; tất cả đến 01/07/2020",
    "Giữ nguyên, không cắt về khoảng thời gian chung", len(m),
    "Cắt đi sẽ bỏ phí dữ liệu thật; để người phân tích tự lọc theo câu hỏi. Kiểm chứng: assert không trùng khoá và không đứt ngày trong notebook 07. "
    "Rủi ro: so sánh giữa các thành phố phải dùng khoảng chung 14/06/2017 - 01/07/2020, nếu không sẽ lệch (Jaipur, Amritsar chỉ có từ 2017).")

neg = int((m[POL] < 0).sum().sum())
add("Tất cả", "11 chất, AQI, AQI_Bucket, City",
    f"{neg} giá trị âm; AQI_Bucket khớp thang CPCB của AQI ở mọi dòng; cột City chỉ có 1 giá trị mỗi thành phố; không có dòng trùng hoàn toàn",
    "Giữ nguyên (không có gì cần sửa)", 0,
    "Đây là kiểm tra hợp lệ và nhất quán (EX4.3). Kiểm chứng: hàm validity_checks / check_consistency / check_duplicates trong utils.py trả về 0 vi phạm.")

## B. Missing ẩn và cột bị bỏ

In [3]:
flat_by_city, flat_by_col = {}, {}
for c in CITIES:
    r = raw[raw.City == c].sort_values("Date").reset_index(drop=True)
    _, fl = flag_flatline_zeros(r, cols=POL + ["Xylene"], min_run=3)
    flat_by_city[c] = sum(v for k, v in fl.items() if k != "Xylene")
    for k, v in fl.items():
        flat_by_col[k] = flat_by_col.get(k, 0) + v
sens = {}
for k in (2, 3, 5, 7):
    t = 0
    for c in CITIES:
        r = raw[raw.City == c].sort_values("Date").reset_index(drop=True)
        _, fl = flag_flatline_zeros(r, cols=POL + ["Xylene"], min_run=k); t += sum(fl.values())
    sens[k] = t
top_cols = ", ".join(f"{k} {vn(v)}" for k, v in sorted(flat_by_col.items(), key=lambda kv: -kv[1]) if v)
add("Tất cả", "CO, NOx, Benzene, Toluene (và Xylene)",
    f"Chuỗi số 0 liên tiếp từ 3 ngày trở lên: {vn(sum(flat_by_city.values()))} ô ở các cột được giữ ({by_city(flat_by_city)}); theo cột: {top_cols} (Xylene sau đó bị bỏ)",
    "Đổi thành NaN (chuỗi 0 liên tục >= 3 ngày)", sum(flat_by_city.values()),
    "Nồng độ khí thật hiếm khi đứng yên đúng 0 nhiều ngày liền; đây là cảm biến treo (sentinel), nếu giữ sẽ kéo trung bình xuống giả tạo. "
    f"Ngưỡng 3 ngày là quy ước của nhóm. Kiểm chứng độ nhạy: đổi ngưỡng sang 2 / 5 / 7 ngày thì số ô bị đổi lần lượt là {vn(sens[2])} / {vn(sens[5])} / {vn(sens[7])} so với {vn(sens[3])} ở 3 ngày, "
    "tức kết quả ít phụ thuộc vào ngưỡng. Phát hiện thật: bước này làm tỉ lệ thiếu của CO, NOx, Benzene, Toluene cao hơn audit đầu tiên. "
    "Rủi ro: chuỗi 0 ngắn (1-2 ngày) vẫn còn, xem dòng về số 0 còn sót.")

xyl_nn = {c: int(raw[raw.City == c].Xylene.notna().sum()) for c in CITIES}
xyl_pct = {c: round(raw[raw.City == c].Xylene.isna().mean() * 100) for c in CITIES}
add("Tất cả", "Xylene",
    "Thiếu % theo thành phố: " + ", ".join(f"{c} {xyl_pct[c]}%" for c in CITIES) + "; 4/8 thành phố thiếu 100%",
    "Bỏ cột Xylene (và Xylene_was_missing, Xylene_is_outlier) ở cả 8 thành phố", sum(xyl_nn.values()),
    "Một cột rỗng hoàn toàn ở một nửa số thành phố thì không so sánh được giữa các thành phố; nhóm trưởng chốt bỏ cho cả 8 để schema đồng nhất khi gộp. "
    f"Kiểm chứng: sau khi bỏ, không file *_clean.csv nào còn cột Xylene*. Rủi ro: mất {vn(sum(xyl_nn.values()))} ô có dữ liệu thật ({by_city(xyl_nn)}); "
    "nếu sau này cần Xylene thì đọc lại từ file gốc và chỉ dùng 4 thành phố có dữ liệu.")

add("Lucknow", "PM10", "Thiếu 100% mọi năm (trạm không đo PM10)", "Bỏ cột PM10; khi gộp, PM10 của Lucknow là NaN và PM10_was_missing = True", int(len(m[m.City == "Lucknow"])),
    "Không có dữ liệu thật để phân tích hay nội suy. Kiểm chứng: raw của Lucknow có 0 giá trị PM10. Rủi ro: mọi phân tích dùng PM10 chỉ có 7 thành phố.")

In [4]:
# --- bằng chứng cơ chế thiếu cho các cột còn thiếu >= 20% sau làm sạch
def nan_runs(isna):
    runs, run = [], 0
    for v in isna.to_numpy():
        if v: run += 1
        elif run: runs.append(run); run = 0
    if run: runs.append(run)
    return runs

rows_mech = []
for c in CITIES:
    g = m[m.City == c].set_index("Date")
    for p in POL:
        if c == "Lucknow" and p == "PM10": continue
        pct = g[p].isna().mean() * 100
        if pct < 20: continue
        runs = nan_runs(g[p].isna()); tot = sum(runs)
        in_long = sum(r for r in runs if r > MAX_GAP) / tot * 100
        yrs = g[p].isna().groupby(g.index.year).mean() * 100
        full_years = [str(y) for y, v in yrs.items() if v >= 80]
        aqi_miss = g.loc[g[p].isna(), "AQI"].mean(); aqi_have = g.loc[g[p].notna(), "AQI"].mean()
        first = g[p].first_valid_index()
        rows_mech.append((c, p, pct, max(runs), in_long, full_years, aqi_miss, aqi_have, first, int(g[p].isna().sum())))

n_cols = len(rows_mech)
n_over = sum(1 for r in rows_mech if r[2] >= MISSING_MAX_PCT)
add("Tất cả", "PM10, NOx, NH3, Benzene, Toluene (cột thiếu >= 20%)",
    f"{n_cols} cặp (thành phố, cột) còn thiếu >= 20% sau làm sạch; " + (f"ở cả {n_cols} cặp, 100%" if min(r[4] for r in rows_mech) >= 99.5 else f"từ {min(r[4] for r in rows_mech):.0f}% đến 100%") + f" số ngày thiếu nằm trong các khoảng liên tục dài hơn {MAX_GAP} ngày; chi tiết từng cặp ở các dòng ngay dưới",
    "Giữ NaN (không nội suy, không điền); giữ cột chỉ báo <cột>_was_missing; cột thiếu >= 60% đánh dấu KHÔNG dùng làm biến mô hình", sum(r[9] for r in rows_mech),
    "Cơ chế thiếu (W3): thiếu theo khối thời gian liên tục (cảm biến chưa lắp hoặc ngừng hoạt động), không rải ngẫu nhiên từng ngày, nên KHÔNG phải MCAR; "
    "phụ thuộc thời điểm và trạm đo, gần với MAR theo thời gian. Nội suy qua khối dài là bịa số, xoá dòng thì mất cả các cột khác, nên chọn giữ NaN. "
    "Theo W3: cột thiếu > 60% thì cân nhắc bỏ cột nhưng giữ cột chỉ báo; ở đây giữ cột vì vẫn có ích cho EDA ở giai đoạn có dữ liệu, chỉ không đưa vào mô hình. "
    "Rủi ro: không loại trừ được MNAR (thiếu vì giá trị quá cao/thấp) vì không có nguồn ngoài; mô hình dùng các cột này phải xử lý NaN trong pipeline (chỉ học trên tập train) hoặc bỏ cột.")

for c, p, pct, longest, in_long, full_years, am, ah, first, n_nan in rows_mech:
    rel = abs(am - ah) / ah * 100
    aqi_txt = (f"AQI trung bình ở ngày thiếu {vn(am)} so với ngày có {vn(ah)} (lệch {rel:.0f}%): " +
               ("thiếu có liên hệ với mức ô nhiễm hoặc thời điểm, tức không phải MCAR" if rel >= 10
                else "gần bằng nhau, không thấy bằng chứng thiếu phụ thuộc vào mức AQI"))
    verdict = ("thiếu >= 60%: KHÔNG dùng làm biến mô hình (quy ước nhóm)" if pct >= MISSING_MAX_PCT
               else "thiếu < 60%: còn dùng được làm biến (quy ước nhóm), cần xử lý NaN trong pipeline")
    add(c, p,
        f"Còn thiếu {vn(pct, 1)}% sau làm sạch; khoảng thiếu liên tục dài nhất {longest} ngày; {in_long:.0f}% số ngày thiếu nằm trong khoảng > {MAX_GAP} ngày"
        + (f"; thiếu >= 80% ở các năm: {', '.join(full_years)}" if full_years else "")
        + f"; ngày có dữ liệu đầu tiên {first:%d/%m/%Y}",
        "Giữ NaN, giữ " + p + "_was_missing", n_nan,
        f"Kiểm chứng: {aqi_txt}. Kết luận: {verdict}.")

## C. Nội suy

In [5]:
imp = {c: int(sum(((m[m.City == c][f"{p}_was_missing"]) & m[m.City == c][p].notna()).sum() for p in POL)) for c in CITIES}

# kiểm chứng: trung bình trước (sau flatline, giá trị quan sát) và sau nội suy
shift = []
for c in CITIES:
    r = raw[raw.City == c].sort_values("Date").reset_index(drop=True)
    r, _ = flag_flatline_zeros(r, cols=POL, min_run=3)
    g = m[m.City == c].reset_index(drop=True)
    for p in POL:
        a, b = r[p].mean(), g[p].mean()
        n_imp_cp = int((g[f"{p}_was_missing"] & g[p].notna()).sum())
        if n_imp_cp >= 5 and r[p].notna().sum() >= 100:
            shift.append((c, p, (b - a) / a * 100))
shift = pd.DataFrame(shift, columns=["c", "p", "pct"])
big = shift[shift.pct.abs() >= 10].sort_values("pct", key=lambda x: -x.abs())
big_txt = ", ".join(f"{r.c} {r.p} {r.pct:+.0f}%" for r in big.itertuples())
add("Tất cả", "11 chất",
    f"{vn(sum(imp.values()))} ô thiếu nằm giữa hai giá trị thật trong khoảng thiếu <= {MAX_GAP} ngày ({by_city(imp)})",
    f"Nội suy tuyến tính theo thời gian chỉ cho khoảng thiếu <= {MAX_GAP} ngày nằm giữa 2 giá trị thật; giữ cột <cột>_was_missing để biết ô nào là nội suy", sum(imp.values()),
    f"Chuỗi thời gian liên tục, khoảng ngắn giữa hai điểm thật nên đường thẳng là ước lượng hợp lý; {MAX_GAP} ngày (~2 tuần) là ngưỡng thận trọng vì nồng độ khí biến động mạnh theo ngày. "
    f"Kiểm chứng: so trung bình trước và sau nội suy của từng cột, độ lệch trung vị chỉ {vn(shift.pct.abs().median(), 1)}% (nhóm Lucknow/Jaipur còn so 4 chiến lược điền trên PM2.5 ở notebook 02, EX3.3, mean chênh khoảng 1 đơn vị). "
    f"Ngoại lệ: {len(big)} cột lệch từ 10% trở lên ({big_txt}); nguyên nhân có vẻ là các khoảng thiếu rơi vào giai đoạn nồng độ cao nên giá trị nội suy cao hơn trung bình cả chuỗi (đã kiểm tra ở Chennai Benzene: trung bình năm 2015 cao gấp nhiều lần các năm sau, xem dòng về Benzene/Toluene), "
    "chưa kiểm tra riêng từng cột còn lại. Đây là hệ quả của việc thiếu tập trung theo thời gian chứ không phải lỗi của phép nội suy, nhưng trung bình toàn chuỗi của các cột này nên tính kèm cờ _was_missing. "
    "Rủi ro khi dự đoán: ô nội suy dùng giá trị của ngày SAU nên nhìn thấy tương lai (rò rỉ); khi đánh giá mô hình phải loại các dòng có <cột>_was_missing = True khỏi tập test hoặc ghi chú.")

fix = {"Delhi": 98, "Gurugram": 294, "Bengaluru": 196, "Amritsar": 252, "Chennai": 252, "Hyderabad": 224}
add("Delhi, Gurugram, Bengaluru, Amritsar, Chennai, Hyderabad", "11 chất",
    f"Lần kiểm tra lại phát hiện {vn(sum(fix.values()))} ô bị điền nhầm trong các khoảng thiếu > {MAX_GAP} ngày ({by_city(fix)}); Lucknow và Jaipur có 0 ô",
    "Sửa hàm nội suy trong src/utils.py; các ô nhầm trả về NaN (notebook 07, Bước 2b); tính lại cờ <cột>_is_outlier", sum(fix.values()),
    f"Quy ước là khoảng thiếu > {MAX_GAP} ngày giữ nguyên NaN, nhưng hàm cũ dùng interpolate(limit=14) của pandas, hàm này điền 14 ngày ĐẦU của mỗi khoảng dài rồi mới dừng, tức vẫn bịa số. "
    "Bạn làm Lucknow/Jaipur phát hiện lỗi ở notebook 02 và dùng hàm riêng nên không bị. "
    "Kiểm chứng: (1) đếm trực tiếp trên file cũ ra đúng số ô trên; (2) chạy lại 01 -> 06 với utils đã sửa cho ra file khác file cũ ở đúng các ô đó; "
    "(3) notebook 07 dựng lại toàn bộ giá trị từ file gốc chỉ bằng quy tắc đã nêu và khớp 100%. Ghi chú: notebook 02 ghi Delhi 168 ô, con số đo ở đây là 98 do đo trực tiếp trên file kết quả. "
    "Đây là một vòng lặp đúng nghĩa của quy trình (kiểm tra lại dẫn đến sửa lại), nên được ghi vào nhật ký.")

aqi_miss = {c: int(m[m.City == c].AQI.isna().sum()) for c in CITIES}
add("Tất cả", "AQI, AQI_Bucket",
    "AQI gốc thiếu " + ", ".join(f"{c} {vn(m[m.City == c].AQI.isna().mean() * 100, 1)}%" for c in CITIES),
    "Không nội suy, không điền AQI; AQI_Bucket để NaN cùng AQI", sum(aqi_miss.values()),
    "AQI nhiều khả năng là biến kết quả của phân tích; điền AQI là tạo ra 'kết quả giả' rồi đem đi phân tích. "
    "Kiểm chứng: 0 dòng gốc có AQI mà thiếu nhãn AQI_Bucket (và ngược lại), nên không phải suy nhãn. "
    "Rủi ro: dòng thiếu AQI phải bỏ hoặc xử lý riêng khi làm mô hình dự đoán AQI; Gurugram thiếu 13,6% nên ảnh hưởng nhiều nhất.")

## D. Ngoại lai và kiểm tra tính nhất quán

In [6]:
flag_cols = [f"{p}_is_outlier" for p in POL]
fl_city = {c: int(m[m.City == c][flag_cols].sum().sum()) for c in CITIES}
pm = m[m["PM2.5_is_outlier"]]
add("Tất cả", "11 chất",
    f"IQR tính riêng theo từng tháng dương lịch (k = 1,5): {vn(sum(fl_city.values()))} ô bị đánh dấu ({by_city(fl_city)}); riêng PM2.5 có {len(pm)} ngày",
    "Giữ nguyên giá trị, chỉ gắn cờ <cột>_is_outlier", sum(fl_city.values()),
    "Đỉnh ô nhiễm mùa đông (đốt rơm rạ, Diwali) là sự kiện thật; tính ngưỡng riêng từng tháng để không gắn nhầm mùa cao điểm, và xoá sẽ làm lệch xu hướng đúng ở chỗ quan trọng nhất. "
    "Hệ số 1,5 và tối thiểu 10 quan sát mỗi tháng là lựa chọn mặc định của IQR, ghi lại để người khác tái tạo được. "
    f"Kiểm chứng: AQI trung vị của các ngày PM2.5 bị cờ là {vn(pm.AQI.median())} so với {vn(m[~m['PM2.5_is_outlier']].AQI.median())} ở ngày không bị cờ, "
    "tức đa số là sự kiện ô nhiễm thật chứ không phải số lạ đứng riêng. Rủi ro: mô hình nhạy với ngoại lai nên so kết quả có và không có các dòng bị cờ.")

for c in CITIES:
    s = sus[(sus.city == c) & (sus.action != "giữ nguyên")]
    if s.empty:
        mo = sorted(sus[sus.city == c].date.dt.month.unique())
        add(c, "PM2.5, AQI", f"Đã điều tra 5 điểm lớn nhất của PM2.5 và 5 của AQI: tất cả rơi vào tháng {', '.join(map(str, mo))} (mùa cao điểm đốt rơm rạ/Diwali), không điểm nào bị luật gợi ý là nghi lỗi cảm biến",
            "Giữ nguyên", int((sus.city == c).sum()),
            "Cô lập, trái mùa và chất khác không tăng theo mới là dấu hiệu lỗi cảm biến; các đỉnh của Delhi đúng mùa cao điểm. "
            f"Kiểm chứng: luật cho kết luận 'thật (mùa cao điểm)' ở cả {int((sus.city == c).sum())} điểm, bằng chứng từng điểm ở reports/suspect_points_8cities.csv. "
            "Rủi ro: luật chỉ xét top 5, chưa xét các đỉnh nhỏ hơn.")
        continue
    p25 = s[s.column == "PM2.5"].sort_values("date"); aq = s[s.column == "AQI"].sort_values("date")
    f25 = "; ".join(f"{d:%d/%m/%Y} ({vn(v, 1)})" for d, v in zip(p25.date, p25.value_original))
    fa = "; ".join(f"{d:%d/%m/%Y} ({vn(v)})" for d, v in zip(aq.date, aq.value_original))
    gmin, gmax = p25.gap_vs_neighbors.min(), p25.gap_vs_neighbors.max()
    pr = p25.pm25_div_pm10.dropna()
    pr_txt = f"; PM2.5/PM10 từ {vn(pr.min(), 1)} đến {vn(pr.max(), 1)}" if len(pr) else ""
    months = sorted(s.date.dt.month.unique())
    n_days = len(set(s.date))
    add(c, "PM2.5, AQI",
        f"{len(p25)} điểm PM2.5 nghi lỗi cảm biến: {f25}" + (f"; AQI cùng ngày: {fa}" if len(aq) else ""),
        "Đặt NaN (không nội suy), AQI_Bucket cùng ngày cũng NaN; gắn cờ suspect_sensor_error", len(p25) + len(aq),
        "Điểm cô lập, trái mùa, chất khác không tăng theo, hoặc PM2.5 vô lý so với PM10 (> 1,5 lần); lỗi cảm biến làm PM2.5 và AQI cùng vọt lên nên AQI cùng ngày cũng bị đặt NaN. "
        f"Kiểm chứng: luật 4 bằng chứng của notebook 02, các điểm PM2.5 cao gấp {vn(gmin, 1)}-{vn(gmax, 1)} lần ngày lân cận{pr_txt}; tháng {', '.join(map(str, months))}; với Lucknow/Jaipur, "
        "chạy lại luật cho ra đúng các ngày nhóm đã xử lý. Chưa đối chiếu được nguồn CPCB nên gọi là NGHI lỗi, không phải đã xác minh; giá trị gốc lưu ở reports/suspect_points_8cities.csv. "
        "Rủi ro: có thể xoá nhầm một đỉnh thật (xem cột suggestion trong file báo cáo để phản biện).")

kept = sus[sus.action == "giữ nguyên"]
add("Tất cả", "PM2.5, AQI",
    f"Đã điều tra tổng cộng {len(sus)} điểm lớn nhất; {len(kept)} điểm được giữ lại vì là đỉnh mùa cao điểm, có chất khác cùng tăng, hoặc 'chưa rõ'",
    "Giữ nguyên (điểm 'chưa rõ' gắn cờ, không xoá)", len(kept),
    "Không đủ bằng chứng để coi là lỗi; xoá nhầm đỉnh thật gây hại hơn giữ nhầm một vài điểm. Quyết định không hành động cũng là một quyết định. Kiểm chứng: từng điểm có một dòng riêng trong reports/suspect_points_8cities.csv (giá trị, bằng chứng, gợi ý, hành động).")

g = m[(m.City == "Gurugram") & (m.Date.between("2017-08-28", "2017-08-31"))].set_index("Date")
add("Gurugram", "PM2.5, AQI",
    f"Điểm đã nhìn thấy nhưng CHƯA xử lý: PM2.5 ngày 30/08/2017 là {vn(g.loc['2017-08-30', 'PM2.5'], 1)} (hai ngày cạnh đó là {vn(g.loc['2017-08-28', 'PM2.5'], 1)} và {vn(g.loc['2017-08-31', 'PM2.5'], 1)}), AQI {vn(g.loc['2017-08-30', 'AQI'])} giữa mùa gió mùa; ngày 29/08 đã bị đặt NaN",
    "Giữ nguyên, ghi vào hạn chế", 1,
    "Luật điều tra chỉ xét 5 điểm lớn nhất mỗi cột nên điểm này (không lọt top 5) bị bỏ sót, dù trông như cùng lỗi với ngày 29/08. "
    "Kiểm chứng: so với hai ngày cạnh; xét top 10 thì luật gợi ý thêm khoảng 26 điểm ở các thành phố nhưng nhiều điểm (Hyderabad, Bengaluru) có thể là đỉnh thật. "
    "Rủi ro: còn vài điểm lỗi rải rác, ảnh hưởng nhỏ lên trung bình nhưng có thể thành đỉnh giả trên biểu đồ; cần người đọc reports/suspect_points_8cities.csv quyết có mở rộng luật không.")

a500 = m[m.AQI > 500]
hi_pm = (a500["PM2.5"] >= 200).mean() * 100
jun = m[m.Date.between("2018-06-14", "2018-06-16") & m.City.isin(["Amritsar", "Delhi"])].dropna(subset=["PM2.5", "PM10"])
ratio_jun = (jun.PM10 / jun["PM2.5"])
add("Delhi, Gurugram, Lucknow, Amritsar, Hyderabad", "AQI",
    f"{len(a500)} ngày AQI > 500 (thang CPCB tối đa 500): {by_city(a500.groupby('City').size().reindex(CITIES).fillna(0).astype(int))}; {int((a500.Date.dt.month.isin([10, 11, 12, 1, 2])).sum())} ngày rơi vào tháng 10-2; {hi_pm:.0f}% số ngày này có PM2.5 >= 200",
    "Giữ nguyên", len(a500),
    "Phần lớn là mùa cao điểm đốt rơm rạ/Diwali ở Delhi và các thành phố lân cận. "
    f"Kiểm chứng: cụm 14-16/06/2018 xảy ra cùng lúc ở Amritsar, Delhi, Gurugram; ở Amritsar và Delhi PM10 gấp {vn(ratio_jun.min(), 1)}-{vn(ratio_jun.max(), 1)} lần PM2.5 (dạng bụi thô), hợp với một sự kiện chung hơn là lỗi riêng của một cảm biến; chưa đối chiếu nguồn ngoài. "
    "Rủi ro: vài ngày AQI quá cao có thể là lỗi, ví dụ Gurugram 30/08/2017 (dòng trên) và Gurugram 12/07/2017 (AQI 576 khi PM2.5 chỉ 75,8, PM10 không có để đối chiếu).")

pp = m.groupby("City").pm25_gt_pm10_flag.agg(lambda s: (int(s.fillna(False).sum()), int(s.notna().sum()))).reindex(CITIES)
add("Tất cả (không có Lucknow)", "PM2.5, PM10",
    "Ngày PM2.5 > 1,2 x PM10 (vô lý vì PM10 đã gồm PM2.5), trên số ngày kiểm tra được: "
    + ", ".join(f"{c} {a}/{b} ({vn(a / b * 100, 1)}%)" for c, (a, b) in pp.items() if b),
    "Giữ nguyên, gắn cờ pm25_gt_pm10_flag", int(sum(a for a, b in pp)),
    "Không xác định được cột nào sai (PM2.5 hay PM10), xoá một trong hai chưa có cơ sở; trước đây chỉ Jaipur kiểm tra, giờ cả 8 thành phố cùng quy tắc. "
    f"Kiểm chứng: Jaipur ra đúng 4 ngày như bạn làm Jaipur đã tính. Chennai cao bất thường ({vn(pp.loc['Chennai'][0] / pp.loc['Chennai'][1] * 100, 1)}% số ngày kiểm tra được; thành phố cao thứ hai là Bengaluru {vn(pp.loc['Bengaluru'][0] / pp.loc['Bengaluru'][1] * 100, 1)}%, các nơi còn lại dưới 2%), chưa rõ nguyên nhân. "
    "Rủi ro: tỉ lệ PM2.5/PM10 ở Chennai không đáng tin; Chennai chỉ có PM10 từ 2019 nên mẫu kiểm tra nhỏ.")

lv = []
for c, p in [("Chennai", "Benzene"), ("Chennai", "Toluene"), ("Bengaluru", "Benzene")]:
    g = m[m.City == c]
    ym = g.groupby(g.Date.dt.year)[p].mean()
    lv.append(f"{c} {p}: " + ", ".join(f"{y} {vn(v, 1)}" for y, v in ym.items()))
add("Chennai, Bengaluru", "Benzene, Toluene",
    "Trung bình theo năm lệch rất mạnh giữa các năm: " + " | ".join(lv) + f"; giá trị lớn nhất của Chennai Benzene là {vn(raw8[raw8.City == 'Chennai'].Benzene.max(), 1)}",
    "Giữ nguyên, ghi vào hạn chế (nhóm Chennai đã ghi nhận ở notebook 06, chưa chốt)", int(m[(m.City == "Chennai")][["Benzene", "Toluene"]].notna().sum().sum()),
    "Mức nồng độ năm đầu cao gấp nhiều lần các năm sau có thể do thay thiết bị/hiệu chuẩn hoặc do thực tế; không có nguồn ngoài để quyết, nên không xoá và không sửa. "
    "Rủi ro: xu hướng theo năm của Benzene/Toluene ở Chennai và Bengaluru bị chi phối bởi năm đầu; khi phân tích dùng trung vị theo năm, hoặc loại các cột này khỏi so sánh xu hướng.")

zeros = (m[POL] == 0).groupby(m.City).sum()
zl = [(c, p, int(zeros.loc[c, p])) for c in CITIES for p in POL if zeros.loc[c, p] >= 30]
add("Chennai, Amritsar", "Benzene, Toluene, CO",
    "Số 0 còn lại sau bước flatline (chuỗi dưới 3 ngày): " + ", ".join(f"{c} {p} {n}" for c, p, n in zl),
    "Giữ nguyên", sum(n for _, _, n in zl),
    "Chuỗi 0 ngắn dưới ngưỡng 3 ngày có thể là giá trị thật rất thấp; chưa đủ bằng chứng để coi là sai. "
    "Rủi ro: có thể vẫn là số 0 giả, nhất là Chennai Benzene/Toluene; phép tính trung bình Benzene của Chennai bị kéo xuống.")

## E. Chuẩn hoá cấu trúc và gộp

In [7]:
add("Lucknow, Jaipur", "PM2.5_iqr_flag, PM2.5_ctx_flag, AQI_iqr_flag, AQI_ctx_flag",
    "Hai thành phố dùng bộ cờ ngoại lai riêng (IQR toàn chuỗi và cờ ngữ cảnh), khác 6 thành phố còn lại (IQR theo tháng)",
    "Bỏ cờ riêng, tính lại <cột>_is_outlier bằng flag_outliers_iqr() của utils.py; giữ suspect_sensor_error", int(len(m[m.City.isin(["Lucknow", "Jaipur"])])),
    "Nếu để nguyên thì bảng gộp lệch cột và hai thành phố được đối xử khác 6 thành phố còn lại. "
    "Kiểm chứng: giá trị nồng độ trong lucknow_clean.csv và jaipur_clean.csv không đổi so với file interim; chỉ cột cờ được tính lại. "
    "Rủi ro: cờ ngữ cảnh (cao gấp >= 2,5 lần lân cận) không còn trong bảng gộp; vẫn có thể tính lại từ gap_utils.contextual_ratio.")

add("Tất cả", "(toàn bộ)",
    f"Gộp 8 thành phố: {vn(len(m))} dòng x {m.shape[1]} cột (Date, City, 11 chất, AQI, AQI_Bucket, 12 cột *_was_missing, 11 cột *_is_outlier, pm25_gt_pm10_flag, suspect_sensor_error)",
    "Gộp bằng pd.concat, thêm 2 cờ dùng chung; không scale, không mã hoá", len(m),
    "Scale/mã hoá phải fit trên tập train lúc làm mô hình (chống rò rỉ), không áp cố định lên file dùng chung. "
    "Kiểm chứng: 10 phép assert trong notebook 07 (số dòng khớp file gốc; không trùng khoá; không âm; AQI_Bucket khớp; giá trị giống hệt từng file thành phố; dựng lại được từ file gốc).")

## F. Bản rút gọn để đưa vào báo cáo

Báo cáo (W8) dành khoảng 2 trang cho nhật ký làm sạch nên bản đầy đủ phía trên (36 dòng) được gom thành **13 dòng**. Vẫn đủ mẫu
`Step · Column · Finding · Action · Rows · Why`, và mỗi `Why` vẫn có đủ **lý do · kiểm chứng · rủi ro**. Số liệu lấy từ cùng các biến đã tính ở trên.
Bản đầy đủ giữ làm phụ lục.

In [8]:
REP = []
def rep_add(column, finding, action, rows, why):
    REP.append(dict(Step=len(REP) + 1, Column=column, Finding=finding, Action=action, Rows=int(rows), Why=why))

per_city_days = sus[sus.action != "giữ nguyên"].groupby("city").date.nunique().reindex(CITIES).fillna(0).astype(int)
over = [r for r in rows_mech if r[2] >= MISSING_MAX_PCT]
rels = [abs(r[6] - r[7]) / r[7] * 100 for r in rows_mech]
chen = pp.loc["Chennai"]; ben = pp.loc["Bengaluru"]
aq_pct = {c: m[m.City == c].AQI.isna().mean() * 100 for c in CITIES}
g830 = m[(m.City == "Gurugram") & (m.Date.between("2017-08-28", "2017-08-31"))].set_index("Date")

rep_add("Cấu trúc dữ liệu",
    f"{vn(len(raw))} dòng gốc, lấy 8 thành phố = {vn(len(raw8))} dòng; 0 dòng trùng (City, Date), không đứt ngày, 0 giá trị âm, AQI_Bucket khớp AQI; độ dài chuỗi khác nhau (Jaipur từ 06/2017, Amritsar từ 02/2017)",
    "Giữ nguyên, không cắt về khoảng thời gian chung; file raw không sửa", len(raw8),
    "Cắt đi sẽ bỏ phí dữ liệu thật. Kiểm chứng: các phép assert ở notebook 07; chạy lại 01 -> 07 từ file gốc ra đúng cùng bảng gộp. Rủi ro: so sánh giữa thành phố phải dùng khoảng chung 14/06/2017 - 01/07/2020.")

rep_add("CO, NOx, Benzene, Toluene (và các cột khác)",
    f"{vn(sum(flat_by_city.values()))} ô là chuỗi số 0 liên tiếp từ 3 ngày trở lên (theo cột: {top_cols})",
    "Đổi thành NaN", sum(flat_by_city.values()),
    f"Nồng độ khí thật hiếm khi đứng đúng 0 nhiều ngày liền; đây là cảm biến treo và sẽ kéo trung bình xuống giả tạo. Kiểm chứng độ nhạy: ngưỡng 2 / 5 / 7 ngày cho {vn(sens[2])} / {vn(sens[5])} / {vn(sens[7])} ô so với {vn(sens[3])} ở 3 ngày, tức ít phụ thuộc vào ngưỡng. Rủi ro: chuỗi 0 ngắn (1-2 ngày) vẫn còn.")

rep_add("Xylene; PM10 (Lucknow)",
    "Xylene thiếu " + ", ".join(f"{c} {xyl_pct[c]}%" for c in CITIES) + "; PM10 của Lucknow thiếu 100%",
    "Bỏ cột Xylene ở cả 8 thành phố và cột PM10 ở Lucknow", sum(xyl_nn.values()),
    f"Cột rỗng ở một nửa số thành phố thì không so sánh được nên bỏ cho cả 8 để schema đồng nhất. Kiểm chứng: không file nào còn cột Xylene*. Rủi ro: mất {vn(sum(xyl_nn.values()))} ô Xylene có dữ liệu thật (Delhi, Gurugram, Amritsar, Hyderabad); PM10 chỉ còn 7 thành phố.")

rep_add("PM10, NOx, NH3, Benzene, Toluene (cột thiếu nhiều)",
    f"{len(rows_mech)} cặp (thành phố, cột) còn thiếu >= 20%; trong đó thiếu >= 60%: " + ", ".join(f"{r[0]} {r[1]} {vn(r[2], 1)}%" for r in over) + ", Lucknow PM10 100%",
    "Giữ NaN (không nội suy qua khối dài), giữ cột <cột>_was_missing; cột thiếu >= 60% không dùng làm biến mô hình", sum(r[9] for r in rows_mech),
    f"Thiếu theo khối thời gian liên tục (cảm biến chưa lắp hoặc ngừng hoạt động): {(("100" if min(r[4] for r in rows_mech) >= 99.5 else f"{min(r[4] for r in rows_mech):.0f}-100"))}% ngày thiếu nằm trong khoảng > {MAX_GAP} ngày, nên không phải MCAR mà gần MAR theo thời gian. Nội suy qua khối dài là bịa số, xoá dòng thì mất cả các cột khác; theo W3, cột thiếu > 60% thì cân nhắc bỏ nhưng giữ cột chỉ báo. "
    f"Kiểm chứng: AQI trung bình giữa ngày thiếu và ngày có lệch từ {min(rels):.0f}% đến {max(rels):.0f}% tuỳ cột. Rủi ro: không loại trừ được MNAR vì không có nguồn ngoài.")

rep_add("11 chất (khoảng thiếu ngắn)",
    f"{vn(sum(imp.values()))} ô thiếu nằm giữa hai giá trị thật trong khoảng thiếu <= {MAX_GAP} ngày",
    f"Nội suy tuyến tính theo thời gian chỉ cho khoảng <= {MAX_GAP} ngày nằm giữa 2 giá trị thật; giữ cờ <cột>_was_missing", sum(imp.values()),
    f"Chuỗi liên tục, khoảng ngắn nên đường thẳng là ước lượng hợp lý; {MAX_GAP} ngày là ngưỡng thận trọng. Kiểm chứng: độ lệch trung vị của trung bình trước và sau nội suy là {vn(shift.pct.abs().median(), 1)}%; {len(big)} cột lệch >= 10% vì khoảng thiếu rơi vào giai đoạn nồng độ cao (đã kiểm tra ở Chennai Benzene). Rủi ro khi dự đoán: ô nội suy dùng giá trị ngày sau (rò rỉ), phải loại các dòng _was_missing = True khỏi tập test.")

rep_add("11 chất (khoảng thiếu dài)",
    f"Kiểm tra lại phát hiện {vn(sum(fix.values()))} ô bị điền nhầm trong khoảng thiếu > {MAX_GAP} ngày ở 6 thành phố ({by_city(fix)}); Lucknow và Jaipur 0 ô",
    "Sửa hàm nội suy trong src/utils.py và trả các ô đó về NaN", sum(fix.values()),
    f"Hàm cũ dùng interpolate(limit=14) điền 14 ngày đầu của khoảng dài rồi mới dừng, trái quy ước không bịa số. Kiểm chứng: đếm trực tiếp trên file cũ; chạy lại 01 -> 06 với hàm đã sửa khác file cũ đúng ở các ô đó; notebook 07 dựng lại toàn bộ giá trị từ file gốc chỉ bằng quy tắc đã nêu và khớp 100%. Đây là một vòng lặp kiểm tra lại rồi sửa lại của quy trình.")

rep_add("AQI, AQI_Bucket",
    f"AQI gốc thiếu từ {vn(min(aq_pct.values()), 1)}% đến {vn(max(aq_pct.values()), 1)}% tuỳ thành phố",
    "Không nội suy AQI; AQI_Bucket để NaN cùng AQI", sum(aqi_miss.values()),
    f"AQI nhiều khả năng là biến kết quả; điền AQI là tạo ra kết quả giả. Kiểm chứng: 0 dòng gốc có AQI mà thiếu nhãn AQI_Bucket và ngược lại. Rủi ro: dòng thiếu AQI phải bỏ khi dự đoán; Gurugram thiếu nhiều nhất ({vn(aq_pct['Gurugram'], 1)}%).")

rep_add("11 chất (ngoại lai)",
    f"IQR tính riêng theo từng tháng (k = 1,5) đánh dấu {vn(sum(fl_city.values()))} ô; riêng PM2.5 có {len(pm)} ngày",
    "Giữ nguyên giá trị, chỉ gắn cờ <cột>_is_outlier", sum(fl_city.values()),
    f"Đỉnh ô nhiễm mùa đông (đốt rơm rạ, Diwali) là sự kiện thật; xoá sẽ làm lệch đúng chỗ quan trọng nhất. Kiểm chứng: AQI trung vị ở các ngày PM2.5 bị cờ là {vn(pm.AQI.median())} so với {vn(m[~m['PM2.5_is_outlier']].AQI.median())} ở ngày khác, tức phần lớn là sự kiện thật. Rủi ro: mô hình nhạy với ngoại lai nên cần so kết quả có và không có các dòng bị cờ.")

rep_add("PM2.5, AQI (nghi lỗi cảm biến)",
    f"Điều tra {len(sus)} điểm lớn nhất (top 5 PM2.5 và top 5 AQI mỗi thành phố): {int(per_city_days.sum())} ngày nghi lỗi cảm biến (" + ", ".join(f"{c} {n}" for c, n in per_city_days.items() if n) + "); Delhi không có",
    f"Đặt NaN (không nội suy), gắn cờ suspect_sensor_error; {len(kept)} điểm còn lại giữ nguyên", int(per_city_days.sum()),
    "Điểm cô lập, trái mùa, chất khác không tăng theo, hoặc PM2.5 > 1,5 lần PM10 là dấu hiệu lỗi cảm biến; đỉnh đúng mùa cao điểm thì giữ. Kiểm chứng: luật 4 bằng chứng của notebook 02 áp đồng nhất cho 8 thành phố, chạy lại ra đúng các ngày Lucknow/Jaipur đã xử lý; mỗi điểm có một dòng trong reports/suspect_points_8cities.csv. Rủi ro: chỉ là nghi lỗi, chưa đối chiếu nguồn CPCB; có thể xoá nhầm một đỉnh thật.")

rep_add("PM2.5, AQI (điểm chưa xử lý)",
    f"Gurugram PM2.5 ngày 30/08/2017 là {vn(g830.loc['2017-08-30', 'PM2.5'], 1)} (hai ngày cạnh {vn(g830.loc['2017-08-28', 'PM2.5'], 1)} và {vn(g830.loc['2017-08-31', 'PM2.5'], 1)}); {len(a500)} ngày AQI > 500 (thang tối đa 500), {int(a500.Date.dt.month.isin([10, 11, 12, 1, 2]).sum())} ngày trong tháng 10-2",
    "Giữ nguyên, ghi vào phần hạn chế", len(a500) + 1,
    "Luật chỉ xét top 5 nên điểm Gurugram không lọt; AQI > 500 phần lớn ở mùa cao điểm, và cụm 14-16/06/2018 xảy ra cùng lúc ở nhiều thành phố nên giống sự kiện chung hơn lỗi riêng. Không có nguồn ngoài để quyết. Rủi ro: còn vài điểm lỗi rải rác có thể thành đỉnh giả trên biểu đồ.")

rep_add("PM2.5, PM10 (nhất quán)",
    f"Ngày PM2.5 > 1,2 x PM10: tổng {int(sum(a for a, b in pp))}; Chennai {vn(chen[0] / chen[1] * 100, 1)}% số ngày kiểm tra được, Bengaluru {vn(ben[0] / ben[1] * 100, 1)}%, các nơi còn lại dưới 2%",
    "Giữ nguyên, gắn cờ pm25_gt_pm10_flag cho cả 8 thành phố", int(sum(a for a, b in pp)),
    "Không xác định được cột nào sai (PM2.5 hay PM10) nên chưa có cơ sở xoá. Kiểm chứng: Jaipur ra đúng 4 ngày như bạn làm Jaipur đã tính riêng. Rủi ro: tỉ lệ PM2.5/PM10 ở Chennai không đáng tin (chỉ có PM10 từ 2019).")

rep_add("Benzene, Toluene, CO (giá trị đáng ngờ nhưng giữ)",
    "Trung bình theo năm lệch rất mạnh: " + " | ".join(lv) + "; số 0 còn sót sau bước flatline: " + ", ".join(f"{c} {p} {n}" for c, p, n in zl),
    "Giữ nguyên, ghi vào phần hạn chế", int(sum(n for _, _, n in zl)),
    "Có thể do đổi thiết bị/hiệu chuẩn hoặc do thực tế, không có nguồn ngoài để quyết; chuỗi 0 ngắn có thể là giá trị thật rất thấp. Rủi ro: xu hướng theo năm của Benzene/Toluene ở Chennai và Bengaluru bị chi phối bởi năm đầu, nên dùng trung vị theo năm hoặc loại khỏi so sánh xu hướng.")

rep_add("Cờ ngoại lai Lucknow/Jaipur; gộp 8 thành phố",
    f"Lucknow và Jaipur dùng bộ cờ ngoại lai riêng; gộp 8 thành phố ra {vn(len(m))} dòng x {m.shape[1]} cột",
    "Tính lại <cột>_is_outlier bằng quy tắc chung; gộp bằng pd.concat; không scale, không mã hoá", len(m),
    "Để cả 8 thành phố cùng một quy tắc; scale và mã hoá phải fit trên tập train lúc làm mô hình (chống rò rỉ). Kiểm chứng: các phép assert (số dòng khớp file gốc, giá trị giống từng file thành phố, dựng lại được từ file gốc). Rủi ro: cờ ngữ cảnh cũ của Lucknow/Jaipur không còn trong bảng gộp.")

report_log = pd.DataFrame(REP)[["Step", "Column", "Finding", "Action", "Rows", "Why"]]
report_log.to_csv(f"{REPORTS}/cleaning_log_report_8cities.csv", index=False, encoding="utf-8-sig")
print("Đã lưu reports/cleaning_log_report_8cities.csv:", len(report_log), "dòng;",
      "số từ trung bình mỗi dòng:", int(report_log[["Finding", "Action", "Why"]].apply(lambda r: sum(len(x.split()) for x in r), axis=1).mean()))
report_log[["Step", "Column", "Rows"]]

Đã lưu reports/cleaning_log_report_8cities.csv: 13 dòng; số từ trung bình mỗi dòng: 103


,Step,Column,Rows
0,1,Cấu trúc dữ liệu,14056
1,2,"CO, NOx, Benzene, Toluene (và các cột khác)",2169
2,3,Xylene; PM10 (Lucknow),4234
3,4,"PM10, NOx, NH3, Benzene, Toluene (cột thiếu nh...",7733
4,5,11 chất (khoảng thiếu ngắn),3329
5,6,11 chất (khoảng thiếu dài),1316
6,7,"AQI, AQI_Bucket",836
7,8,11 chất (ngoại lai),6656
8,9,"PM2.5, AQI (nghi lỗi cảm biến)",24
9,10,"PM2.5, AQI (điểm chưa xử lý)",88


In [9]:
log = pd.DataFrame(LOG)[["Step", "City", "Column", "Finding", "Action", "Rows", "Why"]]
log.to_csv(f"{REPORTS}/cleaning_log_8cities.csv", index=False, encoding="utf-8-sig")
print("Đã lưu reports/cleaning_log_8cities.csv:", len(log), "dòng")
log[["Step", "City", "Column", "Action", "Rows"]]

Đã lưu reports/cleaning_log_8cities.csv: 36 dòng


,Step,City,Column,Action,Rows
0,1,Tất cả,(toàn bộ),"Đọc file gốc, không sửa file raw",14056
1,2,Tất cả,"Date, City","Giữ nguyên, không cắt về khoảng thời gian chung",14056
2,3,Tất cả,"11 chất, AQI, AQI_Bucket, City",Giữ nguyên (không có gì cần sửa),0
3,4,Tất cả,"CO, NOx, Benzene, Toluene (và Xylene)",Đổi thành NaN (chuỗi 0 liên tục >= 3 ngày),2169
4,5,Tất cả,Xylene,"Bỏ cột Xylene (và Xylene_was_missing, Xylene_i...",4234
5,6,Lucknow,PM10,"Bỏ cột PM10; khi gộp, PM10 của Lucknow là NaN ...",2009
6,7,Tất cả,"PM10, NOx, NH3, Benzene, Toluene (cột thiếu >=...","Giữ NaN (không nội suy, không điền); giữ cột c...",7733
7,8,Gurugram,PM10,"Giữ NaN, giữ PM10_was_missing",879
8,9,Gurugram,NOx,"Giữ NaN, giữ NOx_was_missing",399
9,10,Gurugram,NH3,"Giữ NaN, giữ NH3_was_missing",1560
